In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model_path = "../../models/sentiment/bert_binary_weighted"

print("\nModel path exists:", os.path.exists(model_path))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("\nModel loaded successfully. ✅")
print("Number of labels:", model.config.num_labels)
print("Labels:", model.config.id2label)

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU

Model path exists: True


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4441.46it/s]


Model loaded successfully. ✅
Number of labels: 2
Labels: {0: 'negative', 1: 'positive'}


In [2]:
from datasets import load_dataset
import pandas as pd
import re

# Load dataset
dataset = load_dataset("goosmanlei/amazon_reviews_multi", "en")

test_df = dataset["test"].to_pandas()

print("Test shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

# Create binary sentiment label
def create_binary_label(stars):
    if stars <= 3:
        return 0   # negative
    else:
        return 1   # positive

test_df["label"] = test_df["stars"].apply(create_binary_label)

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

test_df["text"] = test_df["review_body"].apply(clean_text)

print("\nLabel distribution:")
print(test_df["label"].value_counts().sort_index())

print("\nSample:")
print(test_df[["text", "stars", "label"]].head())

Test shape: (5000, 8)
Columns: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category']

Label distribution:
label
0    3000
1    2000
Name: count, dtype: int64

Sample:
                                                text  stars  label
0  these are awful. they are see through, the fab...      1      0
1  i bought 4 and none of them worked. yes i used...      1      0
2  on first use it didn't heat up and now it does...      1      0
3  you want an honest answer? i just returned fro...      1      0
4  the glue works fine but the container is impos...      1      0


In [3]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

# Tokenize test texts
encodings = tokenizer(
    test_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

labels = torch.tensor(test_df["label"].values)

# Create DataLoader
test_dataset = TensorDataset(
    encodings["input_ids"],
    encodings["attention_mask"],
    labels
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for input_ids, attention_mask, batch_labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(batch_labels.numpy())

predictions = np.array(predictions)
true_labels = np.array(true_labels)

print("Predictions completed. ✅")
print("Number of predictions:", len(predictions))
print("Unique predictions:", np.unique(predictions))

Predictions completed. ✅
Number of predictions: 5000
Unique predictions: [0 1]


In [4]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(true_labels, predictions)
macro_f1 = f1_score(
    true_labels,
    predictions,
    average="macro"
)

print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        true_labels,
        predictions,
        target_names=["negative", "positive"],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(true_labels, predictions))

Accuracy: 0.8874
Macro F1: 0.8826

Classification Report:
              precision    recall  f1-score   support

    negative     0.9044    0.9083    0.9064      3000
    positive     0.8616    0.8560    0.8588      2000

    accuracy                         0.8874      5000
   macro avg     0.8830    0.8822    0.8826      5000
weighted avg     0.8873    0.8874    0.8873      5000

Confusion Matrix:
[[2725  275]
 [ 288 1712]]


In [5]:
# Add predictions to the test dataframe
test_df["predicted_label"] = predictions

label_names = {
    0: "negative",
    1: "positive"
}

test_df["true_label"] = test_df["label"].map(label_names)
test_df["predicted_label"] = test_df["predicted_label"].map(label_names)

# Get misclassified examples
errors = test_df[
    test_df["true_label"] != test_df["predicted_label"]
].copy()

print("Total errors:", len(errors))
print("\nSample misclassified reviews:\n")

for i, (_, row) in enumerate(errors.head(10).iterrows(), 1):
    print(f"--- Example {i} ---")
    print("True      :", row["true_label"])
    print("Predicted :", row["predicted_label"])
    print("Stars     :", row["stars"])
    print("Review    :", row["text"])
    print()

Total errors: 563

Sample misclassified reviews:

--- Example 1 ---
True      : negative
Predicted : positive
Stars     : 1
Review    : added so much to the projects! works well.

--- Example 2 ---
True      : negative
Predicted : positive
Stars     : 1
Review    : have not used it yet

--- Example 3 ---
True      : negative
Predicted : positive
Stars     : 1
Review    : use at your own risk. my wife and i both had burning sensation and skin reactions. no history of sensitive skin.

--- Example 4 ---
True      : negative
Predicted : positive
Stars     : 1
Review    : haven't used it yet i got it for my son to use

--- Example 5 ---
True      : negative
Predicted : positive
Stars     : 1
Review    : i was impressed by the speed with which it arrived. thank you.

--- Example 6 ---
True      : negative
Predicted : positive
Stars     : 1
Review    : this probably works fine as an ir extender. i bought it on the basis of using it with a otg breakout cable, to control the normally non-ir equ